In [ ]:
from pathlib import Path
import calendar
import json
import logging
import os
import random
import requests
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from datetime import date, datetime, timedelta, timezone

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ---------------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------------
# Final files are stored as:
#   D:/CAMS_modellevel_aerosol_monthly/<variable>/<variable>_ml_YYYYMMDD-YYYYMMDD.grib
# e.g.:
#   D:/CAMS_modellevel_aerosol_monthly/nitrate/nitrate_ml_20160801-20160831.grib

OUTPUT_DIR = Path(r"D:/CAMS_modellevel_aerosol_monthly")
PARTIAL_DIR = OUTPUT_DIR / "_partial"
STATE_DIR = OUTPUT_DIR / "_state"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "cams-global-reanalysis-eac4"
ADS_URL = "https://ads.atmosphere.copernicus.eu/api"

# ---------------------------------------------------------------------------
# Credentials
# ---------------------------------------------------------------------------

DEFAULT_ADS_KEYS = [
    "156656bf-4346-4228-8ea6-a47b8b807223",
    "1e0d795d-e02a-4ebc-81f9-0fb9c07f85a9",
    "1e912241-e383-49bc-a605-4f3155b60ad6",
    "9f9d3677-2705-4764-832b-c5d9bc917eb0",
    "9b7c19b2-d4c7-4ab6-9aef-33d57355862a",
    "2d7328f2-7e18-4fb4-ad55-2ca4605eb1c4",
    "22dec1aa-ecf2-4b87-9641-b80361fdba06",
    "d952344a-3e13-45a6-a7cd-4a7b8e220a79",
    "b6a8b25e-6d7f-46f8-80f6-cf0e37a2719b",
    "40b9f769-d2f6-4369-9b1f-15071345c640",
    "71c70660-fc78-41ce-836b-562ad7f3e1bc",
    "27446185-27c3-4c5d-b178-debf0401be69",
    "54964f95-2780-4c66-b9ef-43e8443d1335",
    "9fd1440b-d970-438c-a31d-1b68807111df",
    "1058bf2f-507b-409e-b072-e36c65f5415d",
    "86dc0e12-c754-4c57-97e9-d7830c599ea9",
    "67bb9bdd-0d37-4576-b45e-80a8f2ff74cc",
    "5916e44d-d2e0-48fe-a032-aea8e6fd4644",
    "bd54dd32-a849-4d68-ba31-3d0f3ad64211",
]

ADS_KEYS = list(DEFAULT_ADS_KEYS)
_env_keys = [x.strip() for x in os.environ.get("ADS_KEYS", "").replace(";", ",").split(",") if x.strip()]
_keys_file = Path.home() / ".ads_keys.txt"
_file_keys = []
if _keys_file.exists():
    _file_keys = [x.strip() for x in _keys_file.read_text(encoding="utf-8").replace(";", ",").split(",") if x.strip()]
ADS_KEYS = list(dict.fromkeys(ADS_KEYS + _env_keys + _file_keys))
if not ADS_KEYS:
    raise RuntimeError("No ADS API keys configured")

cds_file = Path.home() / ".cdsapirc"
cds_file.write_text(f"url: {ADS_URL}\nkey: {ADS_KEYS[0]}\n", encoding="utf-8")

# ---------------------------------------------------------------------------
# Concurrency
# ---------------------------------------------------------------------------

QUEUE_DEPTH_PER_KEY = 5
ACTIVE_DOWNLOADS_PER_KEY = 2
POLL_INTERVAL_SECONDS = 90
POLL_BACKOFF_MAX_SECONDS = 900
SUBMIT_PAUSE_SECONDS = 0.5

REJECTION_COOLDOWN_SECONDS = 60
MAX_REJECTION_COOLDOWN_SECONDS = 600
REJECTION_STREAK_FOR_COOLDOWN_CAP = 4

REJECTION_STREAK_FOR_AUTOWIPE = 8
AUTOWIPE_COOLDOWN_SECONDS = 300

# Pre-flight: at startup, before any new submissions, walk every key's
# server-side job list. For any job in 'successful' state, reconstruct its
# JobSpec from the request metadata and check the local SSD:
#   - If the GRIB already exists at the expected path -> just delete the
#     server-side job (nothing to download).
#   - If the GRIB is missing -> download it FIRST, then delete the job.
#   - If download fails -> leave the job server-side so a later run can rescue it.
# Everything else (failed, rejected, accepted, queued, running, dismissed,
# deleted) gets deleted unconditionally so the per-user cap is freed.
PREFLIGHT_WIPE = True
PREFLIGHT_RESCUE_SUCCESSFUL = True
PREFLIGHT_WIPE_WORKERS_PER_KEY = 8
PREFLIGHT_RESCUE_WORKERS_PER_KEY = 2
PREFLIGHT_RESCUE_DETAIL_WORKERS = 8
PREFLIGHT_MAX_LIST_PAGES = 50
PREFLIGHT_LIST_LIMIT = 200

# ---------------------------------------------------------------------------
# Period
# ---------------------------------------------------------------------------

START_YEAR = 2003
START_MONTH = 8
END_YEAR = 2025
END_MONTH = 8

DATASET_FIRST_DATE = date(2003, 1, 1)
DATASET_LAST_DATE = date(2025, 8, 31)

# ---------------------------------------------------------------------------
# Request scope
# ---------------------------------------------------------------------------

MODEL_LEVELS = [str(i) for i in range(1, 61)]
TIMES = ["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00"]
AREA = [55.7, -11.2, 50.8, -5.2]

PRIORITY_VARIABLES = [
    "methane_sulfonic_acid",
    "ammonium",
    "nitrate",
    "ammonia",
    "dimethyl_sulfide",
]

CHEM_VARIABLES = [
    "methane_sulfonic_acid",
    "ammonium",
    "nitrate",
    "ammonia",
    "dimethyl_sulfide",
    "acetone",
    "acetone_product",
    "aldehydes",
    "amine",
    "dinitrogen_pentoxide",
    "ethanol",
    "ethene",
    "formic_acid",
    "hydroperoxy_radical",
    "lead",
    "methacrolein_mvk",
    "methacrylic_acid",
    "methane_chemistry",
    "methanol",
    "methyl_glyoxal",
    "methyl_peroxide",
    "methylperoxy_radical",
    "nitrate_radical",
    "olefins",
    "organic_ethers",
    "organic_nitrates",
    "paraffins",
    "pernitric_acid",
    "peroxides",
    "peroxy_acetyl_radical",
    "propene",
    "radon",
    "stratospheric_ozone_tracer",
    "terpenes",
]

RUN_PRIORITY_ONLY = False
SHUFFLE_WITHIN_PRIORITY_BLOCKS = True
RANDOM_SEED = 42

# ---------------------------------------------------------------------------
# Retry / validation / timeouts
# ---------------------------------------------------------------------------

MIN_VALID_FILE_BYTES = 1024
MAX_ATTEMPTS_PER_LOGICAL_JOB = 4

MAX_RUNNING_TOTAL_SECONDS = 2 * 3600
MAX_RUNNING_UNCHANGED_SECONDS = 90 * 60
MAX_DOWNLOAD_SECONDS = 45 * 60
MAX_DOWNLOAD_SILENCE_SECONDS = 5 * 60
ACCEPTED_WARN_SECONDS = 2 * 3600
ACCEPTED_DIAG_SECONDS = 4 * 3600
ACCEPTED_HARD_CAP_SECONDS = None

FAILED_RETRY_STATUSES = {"failed", "dismissed", "deleted", "rejected"}
TRANSIENT_HTTP_CODES = {408, 409, 425, 429, 500, 502, 503, 504}
ADMIN_HTTP_TIMEOUT = 30
DOWNLOAD_HTTP_TIMEOUT = 60

NON_RETRY_SUBMIT_SUBSTRINGS = (
    "400 client error",
    "400 bad request",
    "404 client error",
    "404 not found",
)

STATUS_INTERVAL_SECONDS = 60

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

_log_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = OUTPUT_DIR / f"cams_eac4_monthly_scheduler_{_log_stamp}.log"
QUARANTINE_FILE = OUTPUT_DIR / f"cams_eac4_monthly_quarantine_{_log_stamp}.jsonl"
STATS_FILE = OUTPUT_DIR / f"cams_eac4_monthly_stats_{_log_stamp}.jsonl"
CHECKPOINT_FILE = STATE_DIR / f"checkpoint_{_log_stamp}.jsonl"

logger = logging.getLogger("cams_eac4_monthly")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s  %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(_fh)

# ---------------------------------------------------------------------------
# Shared state
# ---------------------------------------------------------------------------

_log_lock = threading.Lock()
_stats_lock = threading.Lock()
_stats = defaultdict(lambda: {
    "submitted": 0, "ok": 0, "skip": 0, "retry": 0,
    "quarantine": 0, "download_err": 0, "poll_err": 0,
    "submit_err": 0, "rejected": 0, "autowipes": 0,
    "server_deletes": 0, "rescued": 0, "rescue_err": 0,
    "rescue_skip_on_disk": 0, "blind_rescued": 0, "blind_unknown": 0,
})
_stop = threading.Event()

_key_cooldown_until = defaultdict(float)
_key_rejection_streak = defaultdict(int)
_key_state_lock = threading.Lock()

_run_start = time.time()


def log_line(x, to_console=True):
    with _log_lock:
        logger.info(x)
        if to_console:
            if HAS_TQDM:
                tqdm.write(x)
            else:
                print(x, flush=True)


def fmt_elapsed(seconds):
    return str(timedelta(seconds=int(seconds)))


def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# ---------------------------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class JobSpec:
    variable: str
    d0: str
    d1: str
    attempt: int = 1


@dataclass
class ServerJob:
    job_id: str
    spec: JobSpec
    key_index: int
    submitted_at: float
    status: str = "submitted"
    status_since: float = 0.0
    last_poll_at: float = 0.0
    next_poll_at: float = 0.0
    poll_interval: float = POLL_INTERVAL_SECONDS
    accepted_warned: bool = False
    accepted_diagged: bool = False
    last_queue_position: int = -1
    last_queue_position_logged_at: float = 0.0


# ---------------------------------------------------------------------------
# Filename helpers (the canonical mapping spec <-> path)
# ---------------------------------------------------------------------------

def parse_date(s):
    return datetime.strptime(s, "%Y-%m-%d").date()


def outfile_for(spec):
    """Canonical output path for a logical job.
    Example: D:/CAMS_modellevel_aerosol_monthly/nitrate/nitrate_ml_20160801-20160831.grib"""
    d0 = parse_date(spec.d0)
    d1 = parse_date(spec.d1)
    return OUTPUT_DIR / spec.variable / f"{spec.variable}_ml_{d0:%Y%m%d}-{d1:%Y%m%d}.grib"


def tmpfile_for(spec, key_index, job_id):
    d0 = parse_date(spec.d0)
    d1 = parse_date(spec.d1)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    return PARTIAL_DIR / (
        f"key{key_index}_{job_id[:8]}_{spec.variable}_ml_"
        f"{d0:%Y%m%d}-{d1:%Y%m%d}_try{spec.attempt}_{stamp}.part.grib"
    )


def file_looks_downloaded(path):
    return path.exists() and path.is_file() and path.stat().st_size > MIN_VALID_FILE_BYTES


def build_request(spec):
    return {
        "variable": [spec.variable],
        "model_level": MODEL_LEVELS,
        "date": f"{spec.d0}/{spec.d1}",
        "time": TIMES,
        "data_format": "grib",
        "area": AREA,
    }


# ---------------------------------------------------------------------------
# ADS HTTP wrappers
# ---------------------------------------------------------------------------

def admin_headers(api_key):
    return {"PRIVATE-TOKEN": api_key}


def process_execute_url():
    return f"{ADS_URL}/retrieve/v1/processes/{DATASET}/execute/"


def job_url(job_id):
    return f"{ADS_URL}/retrieve/v1/jobs/{job_id}"


def jobs_list_url():
    return f"{ADS_URL}/retrieve/v1/jobs"


def job_results_url(job_id):
    return f"{ADS_URL}/retrieve/v1/jobs/{job_id}/results"


def request_is_transient(exc):
    txt = str(exc).lower()
    if any(sub in txt for sub in NON_RETRY_SUBMIT_SUBSTRINGS):
        return False
    tokens = [
        "408", "409", "425", "429", "500", "502", "503", "504",
        "timeout", "timed out", "connection", "temporarily",
        "too many requests", "bad gateway", "service unavailable",
        "remote disconnected", "chunkedencodingerror", "incompleteread",
    ]
    return any(x in txt for x in tokens)


def submit_job(api_key, request):
    r = requests.post(
        process_execute_url(),
        json={"inputs": request},
        headers={**admin_headers(api_key), "Content-Type": "application/json"},
        timeout=ADMIN_HTTP_TIMEOUT,
    )
    if r.status_code in TRANSIENT_HTTP_CODES:
        raise RuntimeError(f"transient submit HTTP {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    body = r.json()
    job_id = body.get("jobID") or body.get("id") or body.get("request_uid")
    if not job_id:
        raise RuntimeError(f"submit returned no jobID: {body}")
    return job_id


def poll_job(api_key, job_id):
    r = requests.get(job_url(job_id), headers=admin_headers(api_key), timeout=ADMIN_HTTP_TIMEOUT)
    if r.status_code in TRANSIENT_HTTP_CODES:
        raise RuntimeError(f"transient poll HTTP {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    body = r.json()
    return str(body.get("status", "")).lower(), body


def get_job_detail(api_key, job_id):
    """Fetch full job detail (includes request inputs in most ADS deployments)."""
    try:
        r = requests.get(job_url(job_id), headers=admin_headers(api_key), timeout=ADMIN_HTTP_TIMEOUT)
    except Exception:
        return None
    if r.status_code == 404:
        return None
    try:
        r.raise_for_status()
        return r.json()
    except Exception:
        return None


def delete_job(api_key, job_id):
    try:
        r = requests.delete(job_url(job_id), headers=admin_headers(api_key), timeout=ADMIN_HTTP_TIMEOUT)
        return r.status_code in (200, 202, 204, 404)
    except Exception:
        return False


def result_href(api_key, job_id):
    r = requests.get(job_results_url(job_id), headers=admin_headers(api_key), timeout=ADMIN_HTTP_TIMEOUT)
    if r.status_code in TRANSIENT_HTTP_CODES:
        raise RuntimeError(f"transient result HTTP {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    body = r.json()
    asset = body.get("asset", {})
    if isinstance(asset, dict):
        value = asset.get("value", {})
        if isinstance(value, dict) and value.get("href"):
            return value["href"], value.get("file:size")
    if body.get("href"):
        return body["href"], None
    raise RuntimeError(f"results response had no href: {body}")


# ---------------------------------------------------------------------------
# Queue-position extraction
# ---------------------------------------------------------------------------

def extract_queue_position(body):
    if not isinstance(body, dict):
        return None
    for k in ("queuePosition", "queue_position", "position", "rank"):
        v = body.get(k)
        if isinstance(v, int) and v >= 0:
            return v
    md = body.get("metadata")
    if isinstance(md, dict):
        for k in ("queuePosition", "queue_position", "position", "rank"):
            v = md.get(k)
            if isinstance(v, int) and v >= 0:
                return v
        msg = md.get("message")
        if isinstance(msg, str):
            n = _scan_position_from_text(msg)
            if n is not None:
                return n
    msg = body.get("message")
    if isinstance(msg, str):
        n = _scan_position_from_text(msg)
        if n is not None:
            return n
    return None


def _scan_position_from_text(text):
    t = text.lower()
    for marker in ("position", "queued at", "rank", "ahead of you", "in queue"):
        if marker in t:
            i = t.find(marker)
            window = t[i:i + 80]
            digits = ""
            in_num = False
            for c in window:
                if c.isdigit():
                    digits += c
                    in_num = True
                elif in_num:
                    break
            if digits:
                try:
                    return int(digits)
                except ValueError:
                    pass
    return None


# ---------------------------------------------------------------------------
# Local checkpoint-based job_id -> JobSpec recovery (PRIMARY path)
# ---------------------------------------------------------------------------
#
# Every time we submit a job we write a 'submitted' line to
# _state/checkpoint_*.jsonl that contains both the job_id we got back and
# the JobSpec we submitted. So at startup we can rebuild a {job_id: JobSpec}
# map from every checkpoint file on disk -- this is far more reliable than
# trying to parse the request payload back out of the ADS job-detail
# response (which doesn't always include it).
#
# The server-side parsing logic that follows is now a FALLBACK for jobs
# that aren't in our local checkpoints (e.g. submitted from another machine
# or before checkpointing was added).

# Used by both the checkpoint loader and the server-side walker to verify
# a candidate variable name is actually one we care about.
_KNOWN_VARIABLES = set(CHEM_VARIABLES)

_checkpoint_index_lock = threading.Lock()
_checkpoint_index = None  # job_id -> dict(spec dict, key_index, last_event, last_time)


def _iter_checkpoint_files():
    """Yield every checkpoint_*.jsonl file under STATE_DIR, oldest first."""
    if not STATE_DIR.exists():
        return
    files = []
    for p in STATE_DIR.glob("checkpoint_*.jsonl"):
        try:
            files.append((p.stat().st_mtime, p))
        except OSError:
            continue
    for _, p in sorted(files):
        yield p


def load_checkpoint_index(verbose=True):
    """Walk every checkpoint_*.jsonl under STATE_DIR and build an in-memory
    map of every job_id we have ever submitted to its JobSpec. Idempotent --
    repeated calls hit the cache."""
    global _checkpoint_index
    with _checkpoint_index_lock:
        if _checkpoint_index is not None:
            return _checkpoint_index
        idx = {}
        files_read = 0
        lines_read = 0
        for path in _iter_checkpoint_files():
            files_read += 1
            try:
                f = open(path, "r", encoding="utf-8-sig", errors="replace")
            except OSError:
                continue
            with f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    lines_read += 1
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    jid = rec.get("job_id")
                    spec = rec.get("spec")
                    if not jid or not isinstance(spec, dict):
                        continue
                    if "variable" not in spec or "d0" not in spec or "d1" not in spec:
                        continue
                    rec_time = rec.get("time", "")
                    existing = idx.get(jid)
                    # Keep the latest record per job_id.
                    if existing is None or rec_time >= existing.get("last_time", ""):
                        idx[jid] = {
                            "variable": spec.get("variable"),
                            "d0": spec.get("d0"),
                            "d1": spec.get("d1"),
                            "attempt": spec.get("attempt", 1),
                            "key_index": rec.get("key_index"),
                            "last_event": rec.get("event"),
                            "last_time": rec_time,
                        }
        _checkpoint_index = idx
        if verbose:
            log_line(
                f"Loaded job-id index from {files_read} checkpoint file(s), "
                f"{lines_read} lines, {len(idx)} unique job_id(s)"
            )
        return idx


def spec_from_checkpoint(job_id):
    """Look up a JobSpec for a job_id we submitted in any prior session.
    Returns None if not found or if the variable isn't one of ours."""
    idx = load_checkpoint_index(verbose=False)
    rec = idx.get(job_id)
    if not rec:
        return None
    variable = rec.get("variable")
    d0 = rec.get("d0")
    d1 = rec.get("d1")
    if not variable or not d0 or not d1:
        return None
    if variable not in _KNOWN_VARIABLES:
        # Variable was renamed or removed from CHEM_VARIABLES -- skip rather
        # than write a file under an unexpected folder.
        return None
    return JobSpec(variable=variable, d0=d0, d1=d1, attempt=1)


# ---------------------------------------------------------------------------
# Spec reconstruction from server-side job metadata (FALLBACK path)
# ---------------------------------------------------------------------------
#
# Walks the entire job-detail JSON tree recursively looking for a dict that
# looks like one of our request payloads (has 'variable' AND 'date'
# parseable). Only used when a job_id isn't in our local checkpoints.

# Where to dump the raw job detail when reconstruction fails. We only write
# the first few so the directory doesn't explode.
DIAG_DIR = STATE_DIR / "unmappable_job_dumps"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
_DIAG_MAX_DUMPS = 8
_diag_dump_lock = threading.Lock()
_diag_dump_count = 0


def _diag_dump(detail, jid, key_index):
    """Write the raw job detail JSON to disk so we can see what shape the
    server returned and extend the extractor if needed. Caps the number of
    dumps so the directory stays manageable."""
    global _diag_dump_count
    with _diag_dump_lock:
        if _diag_dump_count >= _DIAG_MAX_DUMPS:
            return
        _diag_dump_count += 1
        idx = _diag_dump_count
    out = DIAG_DIR / f"unmappable_{idx:02d}_key{key_index}_{jid[:12]}.json"
    try:
        out.write_text(json.dumps(detail, indent=2, default=str), encoding="utf-8")
        log_line(f"DIAG  wrote unmappable job detail to {out}")
    except Exception as exc:
        log_line(f"DIAG  failed to write unmappable dump for {jid[:8]}: {exc}")



def _parse_date_field(date_field):
    """Accept any of:
       'YYYY-MM-DD/YYYY-MM-DD'
       'YYYY-MM-DD' (single day)
       ['YYYY-MM-DD/YYYY-MM-DD']
       ['YYYY-MM-DD', 'YYYY-MM-DD'] (start, end)
       ['YYYY-MM-DD'] (single day)
    Returns (d0, d1) as date objects, or None."""
    if date_field is None:
        return None

    # List inputs
    if isinstance(date_field, list):
        if not date_field:
            return None
        if len(date_field) == 1:
            return _parse_date_field(date_field[0])
        if len(date_field) == 2 and all(isinstance(x, str) for x in date_field):
            try:
                d0 = datetime.strptime(date_field[0].strip(), "%Y-%m-%d").date()
                d1 = datetime.strptime(date_field[1].strip(), "%Y-%m-%d").date()
                return d0, d1
            except ValueError:
                return None
        # Some payloads list every day; treat min/max as the span
        try:
            parsed = [datetime.strptime(str(x).strip(), "%Y-%m-%d").date() for x in date_field]
            return min(parsed), max(parsed)
        except (ValueError, TypeError):
            return None

    if not isinstance(date_field, str):
        return None

    s = date_field.strip()
    if "/" in s:
        d0_str, d1_str = s.split("/", 1)
    else:
        d0_str = d1_str = s
    try:
        d0 = datetime.strptime(d0_str.strip(), "%Y-%m-%d").date()
        d1 = datetime.strptime(d1_str.strip(), "%Y-%m-%d").date()
        return d0, d1
    except ValueError:
        return None


def _normalize_variable(variable):
    """variable may be 'nitrate', ['nitrate'], or even a CSV string."""
    if isinstance(variable, list):
        if len(variable) == 1 and isinstance(variable[0], str):
            return variable[0].strip()
        return None
    if isinstance(variable, str):
        v = variable.strip()
        if "," in v:
            parts = [p.strip() for p in v.split(",") if p.strip()]
            if len(parts) == 1:
                return parts[0]
            return None
        return v
    return None


def _candidate_inputs_from_dict(d):
    """If dict `d` looks like a request payload of ours, return the
    reconstructed (variable, d0, d1). Otherwise None."""
    if not isinstance(d, dict):
        return None
    if "variable" not in d or "date" not in d:
        return None
    variable = _normalize_variable(d.get("variable"))
    if variable is None:
        return None
    if _KNOWN_VARIABLES and variable not in _KNOWN_VARIABLES:
        # Probably some other dict that happens to have these keys.
        return None
    parsed = _parse_date_field(d.get("date"))
    if parsed is None:
        return None
    d0, d1 = parsed
    return variable, d0, d1


def _walk_for_inputs(obj, depth=0, max_depth=12):
    """Depth-first recursive walk for the first dict that looks like a
    request payload (has 'variable' AND 'date' that parse). Bounded depth
    to avoid pathological structures."""
    if depth > max_depth:
        return None
    if isinstance(obj, dict):
        cand = _candidate_inputs_from_dict(obj)
        if cand is not None:
            return cand
        # Visit the likely fast paths first, then everything else.
        preferred = ("inputs", "request", "metadata", "process", "parameters",
                     "body", "spec", "payload")
        seen_keys = set()
        for k in preferred:
            if k in obj:
                seen_keys.add(k)
                got = _walk_for_inputs(obj[k], depth + 1, max_depth)
                if got is not None:
                    return got
        for k, v in obj.items():
            if k in seen_keys:
                continue
            got = _walk_for_inputs(v, depth + 1, max_depth)
            if got is not None:
                return got
        return None
    if isinstance(obj, list):
        for item in obj:
            got = _walk_for_inputs(item, depth + 1, max_depth)
            if got is not None:
                return got
    return None


def spec_from_detail(detail):
    """Convert a full job-detail body (as returned by GET /jobs/{id}) into a
    JobSpec, or return None if the request inputs can't be reconstructed."""
    cand = _walk_for_inputs(detail)
    if cand is None:
        return None
    variable, d0, d1 = cand
    return JobSpec(variable=variable, d0=f"{d0:%Y-%m-%d}", d1=f"{d1:%Y-%m-%d}", attempt=1)


# Backwards-compatible wrappers (older code calls these names)
def _find_request_inputs(detail):
    """Kept for compatibility; returns the candidate dict the new walker would
    have used, reconstructed as a minimal inputs dict."""
    cand = _walk_for_inputs(detail)
    if cand is None:
        return None
    variable, d0, d1 = cand
    return {"variable": [variable], "date": f"{d0:%Y-%m-%d}/{d1:%Y-%m-%d}"}


def spec_from_inputs(inputs):
    """Kept for compatibility with code that already has the inputs dict in hand."""
    cand = _candidate_inputs_from_dict(inputs) if isinstance(inputs, dict) else None
    if cand is None:
        return None
    variable, d0, d1 = cand
    return JobSpec(variable=variable, d0=f"{d0:%Y-%m-%d}", d1=f"{d1:%Y-%m-%d}", attempt=1)


# ---------------------------------------------------------------------------
# Server-side job listing and deletion
# ---------------------------------------------------------------------------

def list_jobs_for_key(api_key, max_pages=PREFLIGHT_MAX_LIST_PAGES, limit=PREFLIGHT_LIST_LIMIT):
    """Return [(job_id, status), ...] for every job this key owns."""
    out = []
    seen = set()
    cursor = None
    for _ in range(max_pages):
        params = {"limit": limit}
        if cursor:
            params["cursor"] = cursor
        try:
            r = requests.get(
                jobs_list_url(),
                headers=admin_headers(api_key),
                params=params,
                timeout=ADMIN_HTTP_TIMEOUT,
            )
        except Exception:
            break
        if r.status_code in (401, 403):
            return []
        try:
            body = r.json() if r.content else {}
        except Exception:
            body = {}
        page = body.get("jobs") or body.get("results") or []
        if not isinstance(page, list):
            page = []
        new = 0
        for job in page:
            if not isinstance(job, dict):
                continue
            jid = job.get("jobID") or job.get("id") or job.get("request_uid")
            if not jid or jid in seen:
                continue
            seen.add(jid)
            out.append((jid, str(job.get("status", "")).lower()))
            new += 1
        nxt = None
        links = body.get("links") or []
        if isinstance(links, list):
            for link in links:
                if isinstance(link, dict) and link.get("rel") == "next":
                    href = link.get("href", "")
                    if "cursor=" in href:
                        nxt = href.split("cursor=", 1)[1].split("&", 1)[0]
                        break
        if nxt and nxt != cursor:
            cursor = nxt
            continue
        if new == 0 or not nxt:
            break
    return out


def wipe_jobs_for_key(api_key, jobs, workers=PREFLIGHT_WIPE_WORKERS_PER_KEY):
    """DELETE every job in `jobs` in parallel. Returns (deleted, already_gone, errors)."""
    deleted = already_gone = errors = 0
    if not jobs:
        return deleted, already_gone, errors

    def _del(jid):
        try:
            r = requests.delete(
                job_url(jid),
                headers=admin_headers(api_key),
                timeout=ADMIN_HTTP_TIMEOUT,
            )
            return r.status_code
        except Exception:
            return -1

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_del, jid) for jid, _ in jobs]
        for fut in as_completed(futures):
            code = fut.result()
            if code in (200, 202, 204):
                deleted += 1
            elif code == 404:
                already_gone += 1
            else:
                errors += 1
    return deleted, already_gone, errors


# ---------------------------------------------------------------------------
# Download primitives (used both at steady state and during pre-flight rescue)
# ---------------------------------------------------------------------------

def stream_download(href, target, tmp):
    started = time.time()
    last_bytes_at = started
    bytes_so_far = 0
    target.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(href, stream=True, timeout=DOWNLOAD_HTTP_TIMEOUT) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                now = time.time()
                if now - started > MAX_DOWNLOAD_SECONDS:
                    raise TimeoutError(f"download exceeded {fmt_elapsed(MAX_DOWNLOAD_SECONDS)} after {bytes_so_far / 1_048_576:.1f} MB")
                if now - last_bytes_at > MAX_DOWNLOAD_SILENCE_SECONDS:
                    raise TimeoutError(f"download silent for {fmt_elapsed(MAX_DOWNLOAD_SILENCE_SECONDS)}")
                if not chunk:
                    continue
                f.write(chunk)
                bytes_so_far += len(chunk)
                last_bytes_at = now
    if not file_looks_downloaded(tmp):
        raise RuntimeError("downloaded file missing or too small")
    tmp.replace(target)
    mb = target.stat().st_size / 1_048_576
    return mb, time.time() - started, total


def download_successful_job(api_key, server_job):
    spec = server_job.spec
    target = outfile_for(spec)
    if file_looks_downloaded(target):
        return "skip", target.stat().st_size / 1_048_576, 0.0
    href, expected_size = result_href(api_key, server_job.job_id)
    tmp = tmpfile_for(spec, server_job.key_index, server_job.job_id)
    try:
        status_tuple = ("ok",) + stream_download(href, target, tmp)
        status, mb, seconds, total = status_tuple
        if expected_size and target.stat().st_size < int(expected_size) * 0.95:
            raise RuntimeError(f"download smaller than expected: {target.stat().st_size} vs {expected_size}")
        # Teach the paramId -> variable mapping while we have a verified
        # (file, variable) pair on hand. Cheap (one GRIB header read).
        try:
            pid, _d = inspect_grib(target)
            if pid is not None:
                remember_paramid(pid, spec.variable)
        except Exception:
            pass
        return status, mb, seconds
    except Exception:
        if tmp.exists() and tmp.stat().st_size > 0:
            failed = tmp.with_suffix(f".failed_{datetime.now():%Y%m%d_%H%M%S}.grib")
            tmp.replace(failed)
        elif tmp.exists():
            tmp.unlink()
        raise


# ---------------------------------------------------------------------------
# GRIB inspection -- identify a downloaded file when we don't know what
# variable/date it is from the request side.
# ---------------------------------------------------------------------------
# We need to recover (paramId, reference_date) from the first GRIB message
# in a file. Two ways:
#   1) Use eccodes if available (clean, robust, handles every edge case).
#   2) Fall back to a pure-bytes parser that reads the first GRIB message's
#      Section 0 + Section 1 (Indicator + Identification) and, if the
#      product needs it, Section 4 for parameterCategory/Number.
# For our purposes we only need to identify each variable once -- after that
# we cache the (paramId -> variable) mapping on disk, so subsequent rescues
# of the same variable skip GRIB inspection entirely.

# Where the learned mapping lives.
PARAMID_MAP_FILE = STATE_DIR / "paramid_to_variable.json"
_paramid_map_lock = threading.Lock()
_paramid_map = None  # str(paramId) -> variable_name


def _load_paramid_map():
    global _paramid_map
    with _paramid_map_lock:
        if _paramid_map is not None:
            return _paramid_map
        m = {}
        if PARAMID_MAP_FILE.exists():
            try:
                m = json.loads(PARAMID_MAP_FILE.read_text(encoding="utf-8"))
                if not isinstance(m, dict):
                    m = {}
            except Exception:
                m = {}
        _paramid_map = m
        return m


def _save_paramid_map():
    global _paramid_map
    with _paramid_map_lock:
        if _paramid_map is None:
            return
        try:
            PARAMID_MAP_FILE.write_text(json.dumps(_paramid_map, indent=2, sort_keys=True), encoding="utf-8")
        except Exception:
            pass


def remember_paramid(paramid, variable):
    """Persistently record that this paramId corresponds to a CAMS variable.
    Future blind-rescue downloads can use this without re-inspecting GRIBs."""
    if paramid is None or not variable:
        return
    if variable not in _KNOWN_VARIABLES:
        return
    m = _load_paramid_map()
    key = str(paramid)
    with _paramid_map_lock:
        if m.get(key) != variable:
            m[key] = variable
            log_line(f"PARAM learned paramId {paramid} -> {variable}")
    _save_paramid_map()


def variable_from_paramid(paramid):
    """Look up a CAMS variable name from a paramId. Uses the learned-on-disk
    mapping (extended every time we successfully rescue a file whose spec was
    known)."""
    if paramid is None:
        return None
    m = _load_paramid_map()
    return m.get(str(paramid))


def seed_paramid_map_from_disk(max_per_variable=6, verbose=True):
    """Populate the paramId -> variable map by inspecting GRIB files that are
    ALREADY on the SSD. Each variable lives in its own folder
    (OUTPUT_DIR/<variable>/...), so the folder name tells us the variable and
    the file's GRIB header tells us the paramId. We only need a couple of
    files per variable to learn its paramId(s), so this is fast even with a
    large archive.

    This is what makes blind-rescue work on the very first run: the map is
    fully populated from existing data before any blind download happens,
    instead of depending on a fresh download occurring during this session."""
    learned_before = len(_load_paramid_map())
    inspected = 0
    variables_seen = 0
    for variable in CHEM_VARIABLES:
        var_dir = OUTPUT_DIR / variable
        if not var_dir.is_dir():
            continue
        # Find candidate GRIBs for this variable.
        count_this_var = 0
        try:
            entries = sorted(var_dir.glob(f"{variable}_ml_*.grib"))
        except OSError:
            entries = []
        if not entries:
            continue
        variables_seen += 1
        for path in entries:
            if count_this_var >= max_per_variable:
                break
            if not file_looks_downloaded(path):
                continue
            try:
                pid, _d = inspect_grib(path)
            except Exception:
                pid = None
            inspected += 1
            if pid is not None:
                remember_paramid(pid, variable)
                count_this_var += 1
    learned_after = len(_load_paramid_map())
    if verbose:
        log_line(
            f"Seeded paramId map from disk: inspected {inspected} file(s) across "
            f"{variables_seen} variable folder(s); map now has {learned_after} "
            f"paramId(s) (was {learned_before})"
        )
    return learned_after


# eccodes is deliberately NOT used: on Windows the `import eccodes` step loads
# a native DLL via ctypes and frequently hard-crashes the interpreter with an
# access violation that try/except cannot catch. We rely solely on the
# pure-bytes GRIB parser below, which needs no native dependency.
_HAVE_ECCODES = False


def _inspect_with_eccodes(path):
    """Disabled on purpose (see note above). Always returns (None, None)."""
    return None, None


def _inspect_with_bytes(path):
    """Pure-byte parser for the first GRIB message. Handles GRIB1 and GRIB2.
    Returns (paramid_str, date_obj) or (None, None) on failure.

    GRIB1 layout (relevant bytes within Section 1):
        octet 4   : table version (tables2Version)
        octet 5   : centre
        octet 9   : indicatorOfParameter
        octets 13-17 : year-of-century, month, day, hour, minute
        octet 25  : century
    For ECMWF, paramId is computed by table2Version * 1000 + indicatorOfParameter
    but the simpler portable form used at ECMWF is "<indicator>.<tablesVersion>".
    We construct paramId as `tablesVersion*1000+indicator` which matches
    ecCodes' canonical paramId for ECMWF parameters in tables 210/215/217 etc.

    GRIB2 layout (Section 1):
        octets 13-19 : year(2), month, day, hour, minute, second
        + Section 4 for parameterCategory/parameterNumber
    For GRIB2 ECMWF parameters paramId is also computable but the byte
    layout is more involved; eccodes is the reliable route. The pure-bytes
    fallback for GRIB2 here is best-effort: we report
    discipline*256*256 + parameterCategory*256 + parameterNumber as a
    placeholder paramId, which won't match the canonical ecCodes paramId
    but is still a stable identifier we can map from learned files. So
    pure-bytes works as a deduper even when it doesn't produce the canonical
    paramId -- as long as the inputs come from the same dataset, all 'nitrate'
    GRIBs will share the same placeholder, all 'ammonium' GRIBs another, etc."""
    try:
        with open(path, "rb") as f:
            header = f.read(16)
            if len(header) < 16 or header[:4] != b"GRIB":
                return None, None
            edition = header[7]
            if edition == 1:
                # Indicator Section is 8 bytes. Section 1 starts at offset 8.
                f.seek(8)
                sec1_len_bytes = f.read(3)
                if len(sec1_len_bytes) < 3:
                    return None, None
                sec1_len = (sec1_len_bytes[0] << 16) | (sec1_len_bytes[1] << 8) | sec1_len_bytes[2]
                rest = f.read(sec1_len - 3)
                if len(rest) < sec1_len - 3:
                    return None, None
                # rest[0]=tablesVersion, rest[1]=centre, rest[2]=gen-proc,
                # rest[3]=grid-def-cat, rest[4]=flag, rest[5]=indicatorOfParameter,
                # rest[6]=indicatorOfTypeOfLevel, rest[7..8]=level value,
                # rest[9]=year-of-century, rest[10]=month, rest[11]=day,
                # rest[12]=hour, rest[13]=minute,
                # rest[14]=indicatorOfUnitOfTimeRange, rest[15]=P1, rest[16]=P2,
                # rest[17]=timeRangeIndicator, ...
                # rest[21]=century (if present, octet 25 = rest[21])
                tables_version = rest[0]
                indicator = rest[5]
                yoc = rest[9]
                month = rest[10]
                day = rest[11]
                century = rest[21] if sec1_len >= 25 and len(rest) >= 22 else 20
                year = (century - 1) * 100 + yoc
                try:
                    d = date(year, month, day)
                except ValueError:
                    d = None
                paramid = tables_version * 1000 + indicator
                return str(paramid), d
            elif edition == 2:
                # Indicator Section is 16 bytes. Section 1 starts at offset 16.
                f.seek(16)
                sec1_len_bytes = f.read(4)
                if len(sec1_len_bytes) < 4:
                    return None, None
                sec1_len = int.from_bytes(sec1_len_bytes, "big")
                if sec1_len < 21:
                    return None, None
                rest = f.read(sec1_len - 4)
                if len(rest) < sec1_len - 4:
                    return None, None
                # rest[0]=sectionNumber(=1), rest[1..2]=centre, rest[3..4]=subCentre,
                # rest[5]=tablesVersion, rest[6]=localTablesVersion,
                # rest[7]=significanceOfReferenceTime,
                # rest[8..9]=year, rest[10]=month, rest[11]=day,
                # rest[12]=hour, rest[13]=minute, rest[14]=second
                if len(rest) < 15:
                    return None, None
                year = (rest[8] << 8) | rest[9]
                month = rest[10]
                day = rest[11]
                try:
                    d = date(year, month, day)
                except ValueError:
                    d = None
                # Now find Section 4 (Product Definition) to get
                # parameterCategory and parameterNumber.
                discipline = header[6]
                # Walk sections starting after Section 1
                pos = 16 + sec1_len
                f.seek(pos)
                param_cat = None
                param_num = None
                for _ in range(8):
                    head = f.read(5)
                    if len(head) < 5:
                        break
                    if head[:4] == b"7777":
                        break
                    sec_len = int.from_bytes(head[:4], "big")
                    sec_num = head[4]
                    if sec_len < 5:
                        break
                    if sec_num == 4:
                        body = f.read(sec_len - 5)
                        # body[0..1] = NV (num coordinate values after template)
                        # body[2..3] = productDefinitionTemplateNumber
                        # body[4]    = parameterCategory
                        # body[5]    = parameterNumber
                        if len(body) >= 6:
                            param_cat = body[4]
                            param_num = body[5]
                        break
                    else:
                        f.seek(pos + sec_len)
                        pos += sec_len
                if param_cat is None or param_num is None:
                    return None, d
                # Build a stable placeholder paramId from
                # (discipline, parameterCategory, parameterNumber).
                placeholder = discipline * 65536 + param_cat * 256 + param_num
                return f"g2_{discipline}_{param_cat}_{param_num}_t{rest[5]}", d
            else:
                return None, None
    except Exception:
        return None, None


def inspect_grib(path):
    """Return (paramid_str, reference_date) for the first GRIB message.
    Tries eccodes first, falls back to pure-byte parser. Either route is
    enough to identify the variable -- when one route produces canonical
    paramIds and the other doesn't, we just learn whichever we saw."""
    if _HAVE_ECCODES:
        pid, d = _inspect_with_eccodes(path)
        if pid is not None:
            return pid, d
    return _inspect_with_bytes(path)


# Where we stash GRIBs we downloaded blind, before we've identified them.
BLIND_STAGING_DIR = OUTPUT_DIR / "_rescue_staging"
BLIND_STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Where files go if we identified the date but not the variable.
UNKNOWN_DIR = OUTPUT_DIR / "_unknown"


def blind_download_and_classify(api_key, job_id, key_index):
    """For a server-side 'successful' job whose JobSpec we couldn't recover
    (not in checkpoint, ADS didn't return inputs):
      1. Fetch the result href.
      2. Download to staging.
      3. Inspect the GRIB to identify (paramId, date).
      4. Look up variable from learned paramId map.
      5. Move to the canonical OUTPUT_DIR/<variable>/<...>.grib path
         (unless that file already exists, in which case discard).
      6. If variable still unknown, leave in _unknown/ for inspection.
    Returns (action, target_path_or_None) where action is one of:
      'rescued_known'  -- placed at canonical path
      'rescued_skip'   -- canonical file already on disk, staged file deleted
      'unknown_paramid'-- variable not in learned map, file left in _unknown/
      'inspect_failed' -- couldn't read GRIB header, file left in _unknown/
      'download_failed'-- download itself failed (raises)"""
    href, _ = result_href(api_key, job_id)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    staged = BLIND_STAGING_DIR / f"blind_key{key_index}_{job_id[:8]}_{stamp}.grib"
    tmp = staged.with_suffix(".part.grib")
    stream_download(href, staged, tmp)
    if not file_looks_downloaded(staged):
        try:
            staged.unlink()
        except OSError:
            pass
        raise RuntimeError("blind download produced no file")

    paramid, d = inspect_grib(staged)
    if paramid is None or d is None:
        UNKNOWN_DIR.mkdir(parents=True, exist_ok=True)
        dest = UNKNOWN_DIR / f"unrecognized_{job_id[:8]}_{stamp}.grib"
        try:
            staged.replace(dest)
        except OSError:
            pass
        return "inspect_failed", dest

    variable = variable_from_paramid(paramid)
    if not variable:
        UNKNOWN_DIR.mkdir(parents=True, exist_ok=True)
        dest = UNKNOWN_DIR / f"paramid_{paramid}_{d:%Y%m%d}_{job_id[:8]}.grib"
        try:
            staged.replace(dest)
        except OSError:
            pass
        return "unknown_paramid", dest

    # Build the canonical target file path. We don't know the month-end
    # date from a single message; the file covers a date range. Use
    # 1st-of-month -> end-of-month based on the date we read.
    first = d.replace(day=1)
    last = date(d.year, d.month, calendar.monthrange(d.year, d.month)[1])
    spec_like = JobSpec(variable=variable, d0=f"{first:%Y-%m-%d}",
                        d1=f"{last:%Y-%m-%d}", attempt=1)
    target = outfile_for(spec_like)

    if file_looks_downloaded(target):
        try:
            staged.unlink()
        except OSError:
            pass
        return "rescued_skip", target

    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        staged.replace(target)
    except OSError as exc:
        log_line(f"WARN blind rescue: failed to move {staged} -> {target}: {exc}")
        return "inspect_failed", staged
    return "rescued_known", target


# ---------------------------------------------------------------------------
# Pre-flight RESCUE: download server-side successful jobs that aren't on disk
# ---------------------------------------------------------------------------

@dataclass
class RescueDecision:
    """What to do with a single 'successful' server-side job after pre-flight checks."""
    job_id: str
    status: str            # always 'successful' here
    spec: "JobSpec | None" # None if we couldn't reconstruct
    target: "Path | None"  # None if spec is None
    on_disk: bool          # True if the GRIB is already on the SSD
    action: str            # 'skip_on_disk' | 'download' | 'unmappable_delete' | 'pending'


def _detail_for_spec(api_key, jid):
    """Try multiple endpoints to get a body we can recover a JobSpec from.
    Returns (detail_or_None, spec_or_None)."""
    # 1) Primary: GET /jobs/{id}
    detail = get_job_detail(api_key, jid)
    spec = spec_from_detail(detail) if detail else None
    if spec is not None:
        return detail, spec

    # 2) Secondary: GET /jobs/{id}/results -- sometimes this carries the
    # original request inputs in its body too. We only do this if the
    # primary failed, to keep startup cost down.
    try:
        r = requests.get(
            job_results_url(jid),
            headers=admin_headers(api_key),
            timeout=ADMIN_HTTP_TIMEOUT,
        )
        if r.ok and r.content:
            try:
                results_body = r.json()
            except Exception:
                results_body = None
            if results_body:
                spec2 = spec_from_detail(results_body)
                if spec2 is not None:
                    return results_body, spec2
                # Merge both so the diag dump captures everything we saw.
                if isinstance(detail, dict):
                    detail = {"detail": detail, "results": results_body}
                else:
                    detail = {"results": results_body}
    except Exception:
        pass

    return detail, None


def _decide_rescue_actions(key_index, api_key, successful_job_ids):
    """For each 'successful' job id:
       1) Look it up in our local checkpoint index (cheap, no network).
       2) If not found there, fall back to GET /jobs/{id} (+ /results)
          and try to parse the request payload out of the server response.
    Then check whether the resulting target file is already on disk.
    Returns a list of RescueDecision."""
    decisions = []
    if not successful_job_ids:
        return decisions

    # ---- Step 1: try the local checkpoint index for everything ----
    ckpt_specs = {}
    needs_server = []
    for jid in successful_job_ids:
        spec = spec_from_checkpoint(jid)
        if spec is not None:
            ckpt_specs[jid] = spec
        else:
            needs_server.append(jid)

    if ckpt_specs:
        log_line(
            f"  key{key_index}: rescue: matched {len(ckpt_specs)}/{len(successful_job_ids)} "
            f"successful job(s) from local checkpoint index"
        )

    # ---- Step 2: only the unknowns hit the server ----
    server_specs = {}
    server_details = {}
    if needs_server:
        log_line(
            f"  key{key_index}: rescue: querying server for "
            f"{len(needs_server)} job(s) not in checkpoint"
        )

        def _lookup(jid):
            return jid, _detail_for_spec(api_key, jid)

        with ThreadPoolExecutor(max_workers=PREFLIGHT_RESCUE_DETAIL_WORKERS) as pool:
            for fut in as_completed([pool.submit(_lookup, jid) for jid in needs_server]):
                jid, (detail, spec) = fut.result()
                server_details[jid] = detail
                if spec is not None:
                    server_specs[jid] = spec

    # ---- Step 3: build decisions ----
    blind_count = 0
    for jid in successful_job_ids:
        spec = ckpt_specs.get(jid) or server_specs.get(jid)
        if spec is None:
            detail = server_details.get(jid)
            if detail is not None:
                _diag_dump(detail, jid, key_index)
            # No spec available. Instead of giving up, mark this job for
            # blind download + GRIB-header identification. We'll figure out
            # the variable from the file itself once we have it.
            blind_count += 1
            decisions.append(RescueDecision(
                job_id=jid, status="successful", spec=None, target=None,
                on_disk=False, action="blind_download",
            ))
            continue
        target = outfile_for(spec)
        if file_looks_downloaded(target):
            decisions.append(RescueDecision(
                job_id=jid, status="successful", spec=spec, target=target,
                on_disk=True, action="skip_on_disk",
            ))
        else:
            decisions.append(RescueDecision(
                job_id=jid, status="successful", spec=spec, target=target,
                on_disk=False, action="download",
            ))
    if blind_count:
        log_line(
            f"  key{key_index}: rescue: {blind_count} job(s) have no recoverable "
            f"spec -- will blind-download and identify from GRIB header"
        )
    return decisions


def _execute_rescue_downloads(key_index, api_key, decisions):
    """Run the actual downloads for decisions that need them.
    Mutates each decision in place to set action to either 'downloaded',
    'blind_rescued_known', 'blind_unknown_paramid', 'blind_inspect_failed'
    or 'download_failed_keep'.
    Returns (rescued_count, error_count)."""
    known_targets = [d for d in decisions if d.action == "download"]
    blind_targets = [d for d in decisions if d.action == "blind_download"]
    if not known_targets and not blind_targets:
        return 0, 0

    def _do_known(d):
        sj = ServerJob(
            job_id=d.job_id,
            spec=d.spec,
            key_index=key_index,
            submitted_at=time.time(),
            status="successful",
            status_since=time.time(),
        )
        try:
            status, mb, seconds = download_successful_job(api_key, sj)
            return d, "known", status, mb, seconds, None
        except Exception as exc:
            return d, "known", "error", 0.0, 0.0, exc

    def _do_blind(d):
        try:
            action, dest = blind_download_and_classify(api_key, d.job_id, key_index)
            mb = dest.stat().st_size / 1_048_576 if dest and dest.exists() else 0.0
            return d, "blind", action, mb, 0.0, dest, None
        except Exception as exc:
            return d, "blind", "error", 0.0, 0.0, None, exc

    rescued = 0
    errors = 0

    # Run known and blind downloads in the same pool, in parallel.
    futures = []
    with ThreadPoolExecutor(max_workers=PREFLIGHT_RESCUE_WORKERS_PER_KEY) as pool:
        for d in known_targets:
            futures.append(pool.submit(_do_known, d))
        for d in blind_targets:
            futures.append(pool.submit(_do_blind, d))

        for fut in as_completed(futures):
            result = fut.result()
            mode = result[1]
            if mode == "known":
                d, _, status, mb, seconds, exc = result
                if exc is not None:
                    errors += 1
                    d.action = "download_failed_keep"
                    log_line(
                        f"  key{key_index}: RESCUE FAIL job {d.job_id[:8]} "
                        f"({d.spec.variable}/{d.spec.d0}->{d.spec.d1}): {exc} "
                        f"-- keeping server-side, will try again next run"
                    )
                else:
                    rescued += 1
                    d.action = "downloaded"
                    log_line(
                        f"  key{key_index}: RESCUED job {d.job_id[:8]} "
                        f"-> {d.target.name} ({mb:.2f} MB in {fmt_elapsed(seconds)}, {status})"
                    )
            else:  # blind
                d, _, action, mb, _seconds, dest, exc = result
                if exc is not None:
                    errors += 1
                    d.action = "download_failed_keep"
                    log_line(
                        f"  key{key_index}: BLIND RESCUE FAIL job {d.job_id[:8]}: {exc} "
                        f"-- keeping server-side, will try again next run"
                    )
                elif action in ("rescued_known", "rescued_skip"):
                    rescued += 1
                    d.action = "blind_rescued_known"
                    d.spec = None  # already moved into place; no further work
                    d.target = dest
                    with _stats_lock:
                        _stats[key_index]["blind_rescued"] += 1
                    verb = "MOVED" if action == "rescued_known" else "DISCARDED (already on disk)"
                    log_line(
                        f"  key{key_index}: BLIND RESCUED job {d.job_id[:8]} "
                        f"({mb:.2f} MB) -- {verb}: {dest}"
                    )
                elif action == "unknown_paramid":
                    d.action = "blind_unknown_paramid"
                    d.target = dest
                    with _stats_lock:
                        _stats[key_index]["blind_unknown"] += 1
                    log_line(
                        f"  key{key_index}: BLIND job {d.job_id[:8]} ({mb:.2f} MB): "
                        f"paramId not in learned map -- file saved to {dest}, "
                        f"keeping server-side until variable is learned"
                    )
                else:  # inspect_failed
                    d.action = "blind_inspect_failed"
                    d.target = dest
                    with _stats_lock:
                        _stats[key_index]["blind_unknown"] += 1
                    log_line(
                        f"  key{key_index}: BLIND job {d.job_id[:8]} ({mb:.2f} MB): "
                        f"GRIB header unreadable -- file saved to {dest}"
                    )

    return rescued, errors


def preflight_for_key(key_index, api_key):
    """Full pre-flight pass for a single key: rescue successful jobs whose
    files aren't on disk, then delete everything that's safe to delete."""
    t0 = time.time()
    jobs = list_jobs_for_key(api_key)
    if not jobs:
        log_line(f"  key{key_index:<2}: no jobs to wipe ({time.time() - t0:.1f}s)")
        return

    by_status = defaultdict(int)
    for _, st in jobs:
        by_status[st or "(unknown)"] += 1
    breakdown = ", ".join(f"{k}={v}" for k, v in sorted(by_status.items(), key=lambda kv: -kv[1]))

    # ---- Rescue phase ----
    rescued = 0
    rescue_errors = 0
    rescue_skipped_on_disk = 0
    rescue_decisions = []
    blind_unknowns = 0
    if PREFLIGHT_RESCUE_SUCCESSFUL:
        successful_ids = [jid for jid, st in jobs if st == "successful"]
        if successful_ids:
            log_line(
                f"  key{key_index:<2}: found={len(jobs)} [{breakdown}] -- "
                f"inspecting {len(successful_ids)} 'successful' job(s) for rescue"
            )
            rescue_decisions = _decide_rescue_actions(key_index, api_key, successful_ids)
            rescue_skipped_on_disk = sum(1 for d in rescue_decisions if d.action == "skip_on_disk")
            if rescue_skipped_on_disk:
                log_line(
                    f"  key{key_index:<2}: {rescue_skipped_on_disk} successful job(s) "
                    f"already on disk -- will delete without downloading"
                )
            to_download = sum(1 for d in rescue_decisions if d.action == "download")
            to_blind = sum(1 for d in rescue_decisions if d.action == "blind_download")
            if to_download or to_blind:
                log_line(
                    f"  key{key_index:<2}: downloading {to_download} known + "
                    f"{to_blind} blind file(s) before wipe"
                )
                rescued, rescue_errors = _execute_rescue_downloads(key_index, api_key, rescue_decisions)
            # Count blind jobs we couldn't fully classify (need to keep server-side)
            blind_unknowns = sum(
                1 for d in rescue_decisions
                if d.action in ("blind_unknown_paramid", "blind_inspect_failed")
            )
        else:
            log_line(f"  key{key_index:<2}: found={len(jobs)} [{breakdown}] -- no 'successful' jobs to rescue")
    else:
        log_line(f"  key{key_index:<2}: found={len(jobs)} [{breakdown}]")

    # ---- Build the wipe list ----
    # Keep server-side any job whose rescue didn't complete cleanly so the
    # next run gets another chance. That covers:
    #   - download_failed_keep      (HTTP/IO error)
    #   - blind_unknown_paramid     (file saved but variable unknown for now)
    #   - blind_inspect_failed      (file saved but GRIB unparseable)
    # Everything else (downloaded, blind_rescued_known, skip_on_disk,
    # non-successful job statuses) gets deleted.
    keep_actions = {"download_failed_keep", "blind_unknown_paramid", "blind_inspect_failed"}
    keep_ids = {d.job_id for d in rescue_decisions if d.action in keep_actions}
    to_wipe = [(jid, st) for jid, st in jobs if jid not in keep_ids]

    deleted, gone, errors = wipe_jobs_for_key(api_key, to_wipe)

    log_line(
        f"  key{key_index:<2}: rescue: downloaded={rescued} "
        f"already_on_disk={rescue_skipped_on_disk} "
        f"blind_unknown={blind_unknowns} errors={rescue_errors}  |  "
        f"wipe: targets={len(to_wipe)} deleted={deleted} already_gone={gone} "
        f"errors={errors} kept={len(keep_ids)} ({time.time() - t0:.1f}s)"
    )

    with _stats_lock:
        _stats[key_index]["rescued"] += rescued
        _stats[key_index]["rescue_err"] += rescue_errors
        _stats[key_index]["rescue_skip_on_disk"] += rescue_skipped_on_disk
        _stats[key_index]["server_deletes"] += deleted

    return {
        "found": len(jobs),
        "rescued": rescued,
        "rescue_errors": rescue_errors,
        "rescue_skipped_on_disk": rescue_skipped_on_disk,
        "deleted": deleted,
        "wipe_errors": errors,
        "kept": len(keep_ids),
    }


def preflight_wipe_all_keys():
    log_line("=" * 78)
    if PREFLIGHT_RESCUE_SUCCESSFUL:
        log_line("Pre-flight: rescue ready-to-download jobs, then wipe the rest")
        log_line(f"           output root: {OUTPUT_DIR}")
        log_line(f"           existing files are NEVER redownloaded")
    else:
        log_line("Pre-flight wipe: clearing server-side jobs for all keys")
    log_line("=" * 78)

    # Pre-load the {job_id: JobSpec} index from every past checkpoint file.
    # This is the primary path for matching server-side 'successful' jobs to
    # the original logical job we submitted -- far more reliable than parsing
    # the request payload back out of an ADS job-detail response.
    if PREFLIGHT_RESCUE_SUCCESSFUL:
        load_checkpoint_index(verbose=True)
        # Seed the paramId -> variable map from files already on the SSD, so
        # the blind-rescue path can identify GRIBs immediately -- without
        # waiting for a fresh download to happen during this session.
        seed_paramid_map_from_disk(verbose=True)

    totals = defaultdict(int)
    for i, key in enumerate(ADS_KEYS):
        result = preflight_for_key(i, key)
        if result:
            for k, v in result.items():
                totals[k] += v

    log_line(
        f"Pre-flight complete: found={totals['found']} "
        f"rescued={totals['rescued']} "
        f"already_on_disk={totals['rescue_skipped_on_disk']} "
        f"rescue_errors={totals['rescue_errors']} "
        f"deleted={totals['deleted']} "
        f"kept_for_later={totals['kept']} "
        f"wipe_errors={totals['wipe_errors']}"
    )
    log_line("=" * 78)


# ---------------------------------------------------------------------------
# Job enumeration
# ---------------------------------------------------------------------------

def monthly_spans(sy, sm, ey, em):
    spans = []
    y, m = sy, sm
    while (y, m) <= (ey, em):
        first = date(y, m, 1)
        last = date(y, m, calendar.monthrange(y, m)[1])
        first = max(first, DATASET_FIRST_DATE)
        last = min(last, DATASET_LAST_DATE)
        if first <= last:
            spans.append((first, last))
        m += 1
        if m == 13:
            m = 1
            y += 1
    return spans


def variable_order():
    priority = [v for v in PRIORITY_VARIABLES if v in CHEM_VARIABLES]
    rest = [v for v in CHEM_VARIABLES if v not in priority]
    if RUN_PRIORITY_ONLY:
        return priority
    return priority + rest


def make_jobs():
    spans = monthly_spans(START_YEAR, START_MONTH, END_YEAR, END_MONTH)
    variables = variable_order()
    pset = set(PRIORITY_VARIABLES)
    priority = []
    rest = []
    for v in variables:
        for d0, d1 in spans:
            spec = JobSpec(v, f"{d0:%Y-%m-%d}", f"{d1:%Y-%m-%d}", 1)
            if v in pset:
                priority.append(spec)
            else:
                rest.append(spec)
    if SHUFFLE_WITHIN_PRIORITY_BLOCKS:
        rng = random.Random(RANDOM_SEED)
        rng.shuffle(priority)
        rng.shuffle(rest)
    return deque(priority + rest)


# ---------------------------------------------------------------------------
# Checkpoint / quarantine
# ---------------------------------------------------------------------------

def checkpoint(event, spec, key_index=None, job_id=None, reason=None, extra=None):
    payload = {
        "time": now_utc(),
        "event": event,
        "key_index": key_index,
        "job_id": job_id,
        "reason": reason,
        "spec": asdict(spec),
        "extra": extra or {},
    }
    with _log_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(payload, sort_keys=True, default=str) + "\n")


def write_quarantine(spec, reason, key_index=None, job_id=None, body=None):
    payload = {
        "time": now_utc(),
        "reason": reason,
        "key_index": key_index,
        "job_id": job_id,
        "spec": asdict(spec),
        "body": body or {},
    }
    with _log_lock:
        with open(QUARANTINE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(payload, sort_keys=True, default=str) + "\n")
    with _stats_lock:
        _stats[key_index]["quarantine"] += 1
    log_line(f"QUAR  key{key_index} {spec.variable}/{spec.d0}->{spec.d1} try{spec.attempt}: {reason}")


def retry_spec(spec):
    return JobSpec(spec.variable, spec.d0, spec.d1, spec.attempt + 1)


def requeue_or_quarantine(job_queue, job_queue_lock, spec, reason, key_index=None, job_id=None, body=None, front=False):
    if spec.attempt >= MAX_ATTEMPTS_PER_LOGICAL_JOB:
        write_quarantine(spec, reason, key_index, job_id, body)
        checkpoint("quarantine", spec, key_index, job_id, reason, body)
        return
    new_spec = retry_spec(spec)
    with job_queue_lock:
        if front:
            job_queue.appendleft(new_spec)
        else:
            job_queue.append(new_spec)
    checkpoint("retry", new_spec, key_index, job_id, reason)
    with _stats_lock:
        _stats[key_index]["retry"] += 1
    log_line(f"RETRY key{key_index} {spec.variable}/{spec.d0}->{spec.d1} try{spec.attempt}->{new_spec.attempt}: {reason}")


def quarantine_now(spec, reason, key_index=None, job_id=None, body=None):
    write_quarantine(spec, reason, key_index, job_id, body)
    checkpoint("quarantine", spec, key_index, job_id, reason, body)


# ---------------------------------------------------------------------------
# Policy gates
# ---------------------------------------------------------------------------

def accepted_policy(server_job, body):
    age = time.time() - server_job.submitted_at
    if server_job.status in ("accepted", "queued", "submitted"):
        if age > ACCEPTED_WARN_SECONDS and not server_job.accepted_warned:
            server_job.accepted_warned = True
            log_line(
                f"WAIT  key{server_job.key_index} {server_job.spec.variable}/"
                f"{server_job.spec.d0}->{server_job.spec.d1} job {server_job.job_id[:8]} "
                f"queued for {fmt_elapsed(age)}; not cancelling"
            )
        if age > ACCEPTED_DIAG_SECONDS and not server_job.accepted_diagged:
            server_job.accepted_diagged = True
            keys = sorted(body.keys()) if isinstance(body, dict) else []
            msg = body.get("message") if isinstance(body, dict) else None
            log_line(
                f"DIAG  key{server_job.key_index} {server_job.spec.variable}/"
                f"{server_job.spec.d0}->{server_job.spec.d1} job {server_job.job_id[:8]} "
                f"queued for {fmt_elapsed(age)} keys={keys} message={msg}"
            )
        if ACCEPTED_HARD_CAP_SECONDS is not None and age > ACCEPTED_HARD_CAP_SECONDS:
            return True, f"accepted hard cap exceeded {fmt_elapsed(ACCEPTED_HARD_CAP_SECONDS)}"
    return False, None


def running_policy(server_job):
    if server_job.status != "running":
        return False, None
    total_age = time.time() - server_job.submitted_at
    running_age = time.time() - server_job.status_since
    if total_age > MAX_RUNNING_TOTAL_SECONDS:
        return True, f"running total exceeded {fmt_elapsed(MAX_RUNNING_TOTAL_SECONDS)}"
    if running_age > MAX_RUNNING_UNCHANGED_SECONDS:
        return True, f"running unchanged exceeded {fmt_elapsed(MAX_RUNNING_UNCHANGED_SECONDS)}"
    return False, None


# ---------------------------------------------------------------------------
# Rejection handling: cooldown + auto-wipe
# ---------------------------------------------------------------------------

def cooldown_for_streak(streak):
    capped = min(streak, REJECTION_STREAK_FOR_COOLDOWN_CAP)
    return min(REJECTION_COOLDOWN_SECONDS * (2 ** (capped - 1)), MAX_REJECTION_COOLDOWN_SECONDS)


def maybe_autowipe_key(key_index, api_key):
    """If this key's rejection streak is high, run a single-key version of
    the pre-flight pass: rescue any 'successful' jobs first, then delete the rest."""
    with _key_state_lock:
        streak = _key_rejection_streak[key_index]
    if streak < REJECTION_STREAK_FOR_AUTOWIPE:
        return False
    log_line(f"AUTOWIPE key{key_index}: streak={streak}, running rescue+wipe")
    try:
        preflight_for_key(key_index, api_key)
    except Exception as exc:
        log_line(f"AUTOWIPE key{key_index}: error: {exc}")

    with _key_state_lock:
        _key_rejection_streak[key_index] = 0
        _key_cooldown_until[key_index] = time.time() + AUTOWIPE_COOLDOWN_SECONDS
    with _stats_lock:
        _stats[key_index]["autowipes"] += 1
    return True


# ---------------------------------------------------------------------------
# Per-key manager
# ---------------------------------------------------------------------------

def key_manager(key_index, api_key, job_queue, job_queue_lock, progress_bar):
    outstanding = {}
    download_pool = ThreadPoolExecutor(max_workers=ACTIVE_DOWNLOADS_PER_KEY)
    download_futures = {}
    try:
        while not _stop.is_set():
            made_progress = False

            # 1. Reap finished downloads.
            for fut in [x for x in download_futures if x.done()]:
                server_job = download_futures.pop(fut)
                spec = server_job.spec
                try:
                    status, mb, seconds = fut.result()
                    if status == "skip":
                        with _stats_lock:
                            _stats[key_index]["skip"] += 1
                        checkpoint("skip", spec, key_index, server_job.job_id)
                        log_line(f"SKIP  key{key_index} {spec.variable}/{spec.d0}->{spec.d1} already exists")
                    else:
                        with _stats_lock:
                            _stats[key_index]["ok"] += 1
                        checkpoint("ok", spec, key_index, server_job.job_id, extra={"mb": mb, "seconds": seconds})
                        log_line(f"OK    key{key_index} {spec.variable}/{spec.d0}->{spec.d1} {mb:.2f} MB in {fmt_elapsed(seconds)}")
                    if delete_job(api_key, server_job.job_id):
                        with _stats_lock:
                            _stats[key_index]["server_deletes"] += 1
                    if HAS_TQDM and progress_bar:
                        progress_bar.update(1)
                except Exception as exc:
                    with _stats_lock:
                        _stats[key_index]["download_err"] += 1
                    delete_job(api_key, server_job.job_id)
                    requeue_or_quarantine(
                        job_queue, job_queue_lock, spec,
                        f"download/result failed: {exc}",
                        key_index, server_job.job_id, front=False,
                    )
                made_progress = True

            # 2. Submit new work, up to QUEUE_DEPTH_PER_KEY, honouring cooldown.
            while len(outstanding) < QUEUE_DEPTH_PER_KEY:
                with _key_state_lock:
                    cd_until = _key_cooldown_until[key_index]
                if cd_until > time.time():
                    break

                with job_queue_lock:
                    if not job_queue:
                        break
                    spec = job_queue.popleft()

                target = outfile_for(spec)
                if file_looks_downloaded(target):
                    with _stats_lock:
                        _stats[key_index]["skip"] += 1
                    checkpoint("skip", spec, key_index)
                    log_line(f"SKIP  key{key_index} {spec.variable}/{spec.d0}->{spec.d1} already exists")
                    if HAS_TQDM and progress_bar:
                        progress_bar.update(1)
                    made_progress = True
                    continue

                try:
                    job_id = submit_job(api_key, build_request(spec))
                    t = time.time()
                    outstanding[job_id] = ServerJob(
                        job_id=job_id,
                        spec=spec,
                        key_index=key_index,
                        submitted_at=t,
                        status="submitted",
                        status_since=t,
                        last_poll_at=0.0,
                        next_poll_at=t + POLL_INTERVAL_SECONDS,
                        poll_interval=POLL_INTERVAL_SECONDS,
                    )
                    with _stats_lock:
                        _stats[key_index]["submitted"] += 1
                    checkpoint("submitted", spec, key_index, job_id)
                    log_line(f"SUBMIT key{key_index} {spec.variable}/{spec.d0}->{spec.d1} try{spec.attempt} job {job_id[:8]} outstanding={len(outstanding)}")
                    time.sleep(SUBMIT_PAUSE_SECONDS)
                    made_progress = True
                except Exception as exc:
                    with _stats_lock:
                        _stats[key_index]["submit_err"] += 1
                    if request_is_transient(exc):
                        requeue_or_quarantine(
                            job_queue, job_queue_lock, spec,
                            f"transient submit failed: {exc}",
                            key_index, front=True,
                        )
                    else:
                        quarantine_now(spec, f"non-retryable submit failed: {exc}", key_index)
                        if HAS_TQDM and progress_bar:
                            progress_bar.update(1)
                    made_progress = True
                    break

            # 3. Poll outstanding jobs.
            now = time.time()
            for job_id, sj in list(outstanding.items()):
                if sj.next_poll_at > now:
                    continue
                try:
                    status, body = poll_job(api_key, job_id)
                    sj.last_poll_at = now
                    sj.next_poll_at = now + sj.poll_interval
                    sj.poll_interval = min(sj.poll_interval + 5, POLL_BACKOFF_MAX_SECONDS)

                    qpos = extract_queue_position(body)
                    if qpos is not None and qpos != sj.last_queue_position:
                        big_drop = sj.last_queue_position > 0 and (sj.last_queue_position - qpos) >= 5
                        time_since = now - sj.last_queue_position_logged_at
                        if sj.last_queue_position == -1 or big_drop or time_since >= 300:
                            log_line(
                                f"QPOS  key{key_index} {sj.spec.variable}/{sj.spec.d0}->{sj.spec.d1} "
                                f"job {job_id[:8]} queue position={qpos} "
                                f"(was {sj.last_queue_position if sj.last_queue_position >= 0 else 'n/a'})"
                            )
                            sj.last_queue_position_logged_at = now
                        sj.last_queue_position = qpos

                    if status != sj.status:
                        log_line(
                            f"STATE key{key_index} {sj.spec.variable}/{sj.spec.d0}->{sj.spec.d1} "
                            f"job {job_id[:8]} {sj.status}->{status} after {fmt_elapsed(now - sj.status_since)}"
                            + (f" qpos={qpos}" if qpos is not None else "")
                        )
                        sj.status = status
                        sj.status_since = now
                        sj.poll_interval = POLL_INTERVAL_SECONDS
                        sj.next_poll_at = now + POLL_INTERVAL_SECONDS

                    cancel_accepted, accepted_reason = accepted_policy(sj, body)
                    if cancel_accepted:
                        delete_job(api_key, job_id)
                        outstanding.pop(job_id, None)
                        requeue_or_quarantine(
                            job_queue, job_queue_lock, sj.spec,
                            accepted_reason, key_index, job_id, body, front=False,
                        )
                        made_progress = True
                        continue

                    cancel_running, running_reason = running_policy(sj)
                    if cancel_running:
                        delete_job(api_key, job_id)
                        outstanding.pop(job_id, None)
                        requeue_or_quarantine(
                            job_queue, job_queue_lock, sj.spec,
                            running_reason, key_index, job_id, body, front=False,
                        )
                        made_progress = True
                        continue

                    if status == "successful":
                        outstanding.pop(job_id, None)
                        with _key_state_lock:
                            _key_rejection_streak[key_index] = 0
                            _key_cooldown_until[key_index] = 0.0
                        fut = download_pool.submit(download_successful_job, api_key, sj)
                        download_futures[fut] = sj
                        made_progress = True
                        continue

                    if status in FAILED_RETRY_STATUSES:
                        outstanding.pop(job_id, None)
                        if status == "rejected":
                            delete_job(api_key, job_id)
                            with _stats_lock:
                                _stats[key_index]["rejected"] += 1
                            with _key_state_lock:
                                _key_rejection_streak[key_index] += 1
                                streak = _key_rejection_streak[key_index]
                                cooldown = cooldown_for_streak(streak)
                                _key_cooldown_until[key_index] = time.time() + cooldown
                            log_line(
                                f"REJECT key{key_index} {sj.spec.variable}/{sj.spec.d0}->{sj.spec.d1} "
                                f"job {job_id[:8]} -- dataset queue cap "
                                f"(streak={streak}, cooldown={fmt_elapsed(cooldown)}); "
                                f"releasing spec back to front of queue"
                            )
                            with job_queue_lock:
                                job_queue.appendleft(sj.spec)
                            maybe_autowipe_key(key_index, api_key)
                        else:
                            delete_job(api_key, job_id)
                            requeue_or_quarantine(
                                job_queue, job_queue_lock, sj.spec,
                                f"server status={status}", key_index, job_id, body,
                                front=False,
                            )
                        made_progress = True
                        continue

                except Exception as exc:
                    with _stats_lock:
                        _stats[key_index]["poll_err"] += 1
                    sj.next_poll_at = time.time() + min(sj.poll_interval * 2, POLL_BACKOFF_MAX_SECONDS)
                    sj.poll_interval = min(sj.poll_interval * 2, POLL_BACKOFF_MAX_SECONDS)
                    log_line(f"WARN  key{key_index} poll failed job {job_id[:8]} {sj.spec.variable}/{sj.spec.d0}->{sj.spec.d1}: {exc}")
                    made_progress = True

            with job_queue_lock:
                empty = not job_queue
            if empty and not outstanding and not download_futures:
                break

            if not made_progress:
                time.sleep(2)
    finally:
        download_pool.shutdown(wait=True)


# ---------------------------------------------------------------------------
# Status printer (every minute)
# ---------------------------------------------------------------------------

def status_printer(job_queue, job_queue_lock, total_jobs):
    while not _stop.wait(STATUS_INTERVAL_SECONDS):
        with _stats_lock:
            stats = {k: dict(v) for k, v in _stats.items()}
        with job_queue_lock:
            remaining = len(job_queue)
        with _key_state_lock:
            cooldowns = dict(_key_cooldown_until)
            streaks = dict(_key_rejection_streak)
        now = time.time()
        elapsed = now - _run_start

        total_ok = sum(s.get("ok", 0) for s in stats.values())
        total_skip = sum(s.get("skip", 0) for s in stats.values())
        total_submitted = sum(s.get("submitted", 0) for s in stats.values())
        total_rejected = sum(s.get("rejected", 0) for s in stats.values())
        total_autowipes = sum(s.get("autowipes", 0) for s in stats.values())
        total_server_deletes = sum(s.get("server_deletes", 0) for s in stats.values())
        total_rescued = sum(s.get("rescued", 0) for s in stats.values())
        done = total_ok + total_skip
        rate_per_min = (done / elapsed * 60) if elapsed > 0 else 0
        eta_secs = (total_jobs - done) / (done / elapsed) if done > 0 else None
        eta_str = fmt_elapsed(eta_secs) if eta_secs else "—"

        lines = [
            "",
            "-- Scheduler status ------------------------------------------",
            f"  elapsed: {fmt_elapsed(elapsed)}   "
            f"done: {done}/{total_jobs}  rate: {rate_per_min:.1f}/min  ETA: {eta_str}",
            f"  remaining in queue: {remaining}   "
            f"submitted: {total_submitted}   rejected: {total_rejected}   "
            f"rescued: {total_rescued}   autowipes: {total_autowipes}   "
            f"server_deletes: {total_server_deletes}",
        ]
        for k in sorted(stats):
            s = stats[k]
            streak = streaks.get(k, 0)
            cd_remaining = max(0, cooldowns.get(k, 0) - now)
            cd_str = f"  cooldown={fmt_elapsed(cd_remaining)}" if cd_remaining > 0 else ""
            streak_str = f"  reject_streak={streak}" if streak else ""
            lines.append(
                f"  key{k}: ok={s.get('ok', 0):<4} skip={s.get('skip', 0):<3} "
                f"submitted={s.get('submitted', 0):<4} rejected={s.get('rejected', 0):<3} "
                f"rescued={s.get('rescued', 0):<3} retry={s.get('retry', 0):<3} "
                f"quar={s.get('quarantine', 0):<3} deletes={s.get('server_deletes', 0):<4} "
                f"submit_err={s.get('submit_err', 0):<2} poll_err={s.get('poll_err', 0):<2} "
                f"download_err={s.get('download_err', 0):<2}"
                f"{streak_str}{cd_str}"
            )
        lines.append("--------------------------------------------------------------")
        log_line("\n".join(lines))


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    jobs = make_jobs()
    total = len(jobs)
    spans = monthly_spans(START_YEAR, START_MONTH, END_YEAR, END_MONTH)
    variables = variable_order()
    header = [
        "=" * 78,
        "CAMS EAC4 monthly ADS scheduler",
        "=" * 78,
        f"Dataset                  : {DATASET}",
        f"ADS keys                 : {len(ADS_KEYS)}",
        f"Period (requested)       : {START_YEAR}-{START_MONTH:02d} -> {END_YEAR}-{END_MONTH:02d}",
        f"Dataset coverage clip    : {DATASET_FIRST_DATE} -> {DATASET_LAST_DATE}",
        f"Months                   : {len(spans)}",
        f"Variables                : {len(variables)}",
        f"Priority variables       : {', '.join([v for v in PRIORITY_VARIABLES if v in variables])}",
        f"Logical jobs             : {total}",
        f"Queue depth/key          : {QUEUE_DEPTH_PER_KEY}",
        f"Download workers/key     : {ACTIVE_DOWNLOADS_PER_KEY}",
        f"Max outstanding ADS jobs : {QUEUE_DEPTH_PER_KEY * len(ADS_KEYS)}",
        f"Max simultaneous downloads: {ACTIVE_DOWNLOADS_PER_KEY * len(ADS_KEYS)}",
        f"Pre-flight wipe          : {PREFLIGHT_WIPE}",
        f"Pre-flight rescue        : {PREFLIGHT_RESCUE_SUCCESSFUL}",
        f"Auto-wipe streak trigger : {REJECTION_STREAK_FOR_AUTOWIPE} rejections",
        f"Rejection cooldown       : base {fmt_elapsed(REJECTION_COOLDOWN_SECONDS)} "
        f"(cap doubling at streak {REJECTION_STREAK_FOR_COOLDOWN_CAP}, max {fmt_elapsed(MAX_REJECTION_COOLDOWN_SECONDS)})",
        f"Model levels             : {len(MODEL_LEVELS)}",
        f"Times/day                : {len(TIMES)}",
        f"Running total timeout    : {fmt_elapsed(MAX_RUNNING_TOTAL_SECONDS)}",
        f"Running unchanged timeout: {fmt_elapsed(MAX_RUNNING_UNCHANGED_SECONDS)}",
        f"Download timeout         : {fmt_elapsed(MAX_DOWNLOAD_SECONDS)}",
        f"Download silence timeout : {fmt_elapsed(MAX_DOWNLOAD_SILENCE_SECONDS)}",
        f"Max attempts/logical job : {MAX_ATTEMPTS_PER_LOGICAL_JOB}",
        f"Status print interval    : {STATUS_INTERVAL_SECONDS}s",
        f"Output folder            : {OUTPUT_DIR}",
        f"Log file                 : {LOG_FILE}",
        f"Checkpoint file          : {CHECKPOINT_FILE}",
        f"Quarantine file          : {QUARANTINE_FILE}",
        "=" * 78,
    ]
    log_line("\n".join(header))

    # Pre-flight: rescue any 'successful' server jobs whose files aren't on
    # disk, then wipe the rest of each key's server-side queue.
    if PREFLIGHT_WIPE:
        preflight_wipe_all_keys()

    job_queue_lock = threading.Lock()
    bar = tqdm(total=total, unit="file", desc="Monthly files", dynamic_ncols=True) if HAS_TQDM else None
    printer = threading.Thread(target=status_printer, args=(jobs, job_queue_lock, total), daemon=True)
    printer.start()

    try:
        threads = []
        for key_index, api_key in enumerate(ADS_KEYS):
            t = threading.Thread(target=key_manager, args=(key_index, api_key, jobs, job_queue_lock, bar), daemon=False)
            threads.append(t)
            t.start()
        for t in threads:
            t.join()
    finally:
        _stop.set()
        if bar:
            bar.close()

    with _stats_lock:
        stats = {k: dict(v) for k, v in _stats.items()}

    with open(STATS_FILE, "w", encoding="utf-8") as f:
        for k in sorted(stats):
            f.write(json.dumps({"key": k, **stats[k]}, sort_keys=True) + "\n")

    total_submit = sum(s.get("submitted", 0) for s in stats.values())
    total_ok = sum(s.get("ok", 0) for s in stats.values())
    total_skip = sum(s.get("skip", 0) for s in stats.values())
    total_retry = sum(s.get("retry", 0) for s in stats.values())
    total_quar = sum(s.get("quarantine", 0) for s in stats.values())
    total_reject = sum(s.get("rejected", 0) for s in stats.values())
    total_autowipes = sum(s.get("autowipes", 0) for s in stats.values())
    total_server_deletes = sum(s.get("server_deletes", 0) for s in stats.values())
    total_rescued = sum(s.get("rescued", 0) for s in stats.values())
    total_rescue_err = sum(s.get("rescue_err", 0) for s in stats.values())
    total_rescue_skip = sum(s.get("rescue_skip_on_disk", 0) for s in stats.values())

    summary = [
        "",
        "=" * 78,
        "Summary",
        "=" * 78,
        f"Submitted server jobs    : {total_submit}",
        f"Rejected (queue cap)     : {total_reject}",
        f"Server deletes           : {total_server_deletes}",
        f"Auto-wipes triggered     : {total_autowipes}",
        f"Pre-flight rescued       : {total_rescued}",
        f"Pre-flight already on SSD: {total_rescue_skip}",
        f"Pre-flight rescue errors : {total_rescue_err}",
        f"Downloaded OK            : {total_ok}",
        f"Skipped existing         : {total_skip}",
        f"Retries generated        : {total_retry}",
        f"Quarantined              : {total_quar}",
        f"Elapsed wall time        : {fmt_elapsed(time.time() - _run_start)}",
        f"Log file                 : {LOG_FILE}",
        f"Checkpoint file          : {CHECKPOINT_FILE}",
        f"Quarantine file          : {QUARANTINE_FILE}",
        f"Stats file               : {STATS_FILE}",
        "=" * 78,
    ]
    log_line("\n".join(summary))


if __name__ == "__main__":
    main()

CAMS EAC4 monthly ADS scheduler
Dataset                  : cams-global-reanalysis-eac4
ADS keys                 : 19
Period (requested)       : 2003-08 -> 2025-08
Dataset coverage clip    : 2003-01-01 -> 2025-08-31
Months                   : 265
Variables                : 34
Priority variables       : methane_sulfonic_acid, ammonium, nitrate, ammonia, dimethyl_sulfide
Logical jobs             : 9010
Queue depth/key          : 5
Download workers/key     : 2
Max outstanding ADS jobs : 95
Max simultaneous downloads: 38
Pre-flight wipe          : True
Pre-flight rescue        : True
Auto-wipe streak trigger : 8 rejections
Rejection cooldown       : base 0:01:00 (cap doubling at streak 4, max 0:10:00)
Model levels             : 60
Times/day                : 8
Running total timeout    : 2:00:00
Running unchanged timeout: 1:30:00
Download timeout         : 0:45:00
Download silence timeout : 0:05:00
Max attempts/logical job : 4
Status print interval    : 60s
Output folder            : D:\CAMS_

Monthly files:   0%|          | 7/9010 [00:00<04:26, 33.80file/s]

SKIP  key6 nitrate/2025-08-01->2025-08-31 already exists
SKIP  key0 ammonium/2010-05-01->2010-05-31 already exists
SKIP  key8 ammonia/2007-11-01->2007-11-30 already exists
SKIP  key13 dimethyl_sulfide/2006-12-01->2006-12-31 already exists
SKIP  key15 methane_sulfonic_acid/2022-02-01->2022-02-28 already exists
SKIP  key16 dimethyl_sulfide/2014-05-01->2014-05-31 already exists
SKIP  key1 nitrate/2024-09-01->2024-09-30 already exists
SKIP  key3 nitrate/2023-07-01->2023-07-31 already exists


Monthly files:   0%|          | 13/9010 [00:00<02:24, 62.26file/s]

SKIP  key4 nitrate/2020-02-01->2020-02-29 already exists
SKIP  key5 ammonia/2013-02-01->2013-02-28 already exists
SKIP  key9 ammonium/2014-12-01->2014-12-31 already exists
SKIP  key10 methane_sulfonic_acid/2023-05-01->2023-05-31 already exists
SKIP  key11 ammonium/2021-09-01->2021-09-30 already exists
SKIP  key14 ammonium/2020-03-01->2020-03-31 already exists


...

STATE key0 ammonia/2003-12-01->2003-12-31 job 38bdc8db accepted->running after 1:19:16


Monthly files:  10%|▉         | 877/9010 [5:12:49<934:50:33, 413.80s/file]

WARN  key7 poll failed job 04cdaf43 ammonium/2004-12-01->2004-12-31: 404 Client Error: Not Found for url: https://ads.atmosphere.copernicus.eu/api/retrieve/v1/jobs/04cdaf43-5b86-41b1-8652-c600499de82f


Monthly files:  10%|▉         | 877/9010 [5:13:04<934:50:33, 413.80s/file]


-- Scheduler status ------------------------------------------
  elapsed: 5:18:11   done: 877/9010  rate: 2.8/min  ETA: 2 days, 1:10:50
  remaining in queue: 8038   submitted: 255   rejected: 0   rescued: 59   autowipes: 0   server_deletes: 265
  key0: ok=1    skip=55  submitted=15   rejected=0   rescued=6   retry=9   quar=0   deletes=17   submit_err=0  poll_err=0  download_err=0 
  key1: ok=2    skip=54  submitted=15   rejected=0   rescued=4   retry=8   quar=0   deletes=7    submit_err=0  poll_err=0  download_err=0 
  key2: ok=0    skip=46  submitted=10   rejected=0   rescued=3   retry=5   quar=0   deletes=92   submit_err=0  poll_err=0  download_err=0 
  key3: ok=0    skip=17  submitted=10   rejected=0   rescued=1   retry=5   quar=0   deletes=69   submit_err=0  poll_err=0  download_err=0 
  key4: ok=0    skip=59  submitted=12   rejected=0   rescued=4   retry=7   quar=0   deletes=4    submit_err=0  poll_err=0  download_err=0 
  key5: ok=0    skip=27  submitted=17   rejected=0   rescue